In [1]:
!unzip -q -o ./data/you-tube-comments-signal-vsosai.zip -d ./data/

In [2]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

## Dependencies

In [ ]:
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

import matplotlib.pyplot as plt
import seaborn

import emoji

from tqdm.cli import tqdm
from tqdm import trange

tqdm.pandas()

import string

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('names', quiet=True)


True

In [4]:
df_train = pd.read_csv('./data/train.csv')
df_test = pd.read_csv('./data/test.csv')
df_train.head()

,id,comment_text,target
0,59,Eh c kii ce favoritisme lui il a tou led iphon...,1
1,167,It's a myth that Neanderthals were stupid. The...,1
2,169,Emma is super white.\nIn other news...,1
3,372,Chelsea and Aubrey are my favourite on Teen Mo...,1
4,373,that mustard colour tho 😻😻 p.s absolutely love...,1


## EDA

In [5]:
df_train.shape

(51153, 3)

In [6]:
df_train['target'].value_counts()

target
0    38530
1     7808
2     3125
3     1690
Name: count, dtype: int64

In [7]:
df_train['comment_text'][:10]

0    Eh c kii ce favoritisme lui il a tou led iphon...
1    It's a myth that Neanderthals were stupid. The...
2               Emma is super white.\nIn other news...
3    Chelsea and Aubrey are my favourite on Teen Mo...
4    that mustard colour tho 😻😻 p.s absolutely love...
5    OMG u tagged Dina, never see anyone mention he...
6    when you said you're girl twins would be calle...
7                                 Loved this lousie♥♥♥
8    I'm a man united fan but anyone who can't see ...
9    When Christensen speaks better English than Ro...
Name: comment_text, dtype: object

In [8]:
df_train.isna().sum()

id              0
comment_text    9
target          0
dtype: int64

In [9]:
df_train = df_train.fillna("NA")

## Solutions

In [10]:
X = df_train.drop(columns=['id', 'target'])
y = df_train['target']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [11]:
X_train

,comment_text
29554,Fuck life.
44479,woooow
28131,This so so funny
41042,H
34839,Hi
...,...
8444,Clark Gable and George Clooney look the same
31718,The chef is french ?
9697,Pink's reaction would totally be my reaction. ...
37972,ONE HOUR LATER


### CatBoost

In [12]:
model = CatBoostClassifier(
    iterations=50000,
    learning_rate=0.01,
    eval_metric='MultiClass',
    random_seed=42,
    early_stopping_rounds=1000,
    use_best_model=True,
    thread_count=-1,
    task_type='GPU',
    auto_class_weights='Balanced',
    text_features=['comment_text']
)

#model.fit(
#    X_train,
#    y_train,
#    eval_set=(X_val, y_val),
#    verbose=1000
#)

In [13]:
lm = pd.read_csv('./data/label_mapping.csv')
lm = lm['target_name'].to_dict()
lm

{0: 'ignored', 1: 'applause', 2: 'debate', 3: 'bait'}

In [14]:
df_test.head()

,id,comment_text
0,200,Me with Arabic. I can never have a conversatio...
1,607,if I could subscribe to Dude Perfect a million...
2,1100,We can fortunately say that we lived in the er...
3,1137,"Not an easy Classic to cover, you and your ban..."
4,1139,"I really really really love this cover, hope H..."


In [15]:
df_test['comment_text'].isna().sum()

np.int64(3)

In [16]:
df_test['comment_text'] = df_test['comment_text'].fillna('NA')

In [17]:
#subm = df_test.copy()
#subm['pred'] = model.predict(df_test[['comment_text']])
#subm.head()

In [18]:
#subm['target'] = subm['pred']
#subm[['id', 'target']].to_csv("subm.csv", index=False)
#subm.head()

### Linear Models

#### Data preparing

In [19]:
df_train.head()

,id,comment_text,target
0,59,Eh c kii ce favoritisme lui il a tou led iphon...,1
1,167,It's a myth that Neanderthals were stupid. The...,1
2,169,Emma is super white.\nIn other news...,1
3,372,Chelsea and Aubrey are my favourite on Teen Mo...,1
4,373,that mustard colour tho 😻😻 p.s absolutely love...,1


In [25]:
df_extra = pd.read_csv('./data/extra_unlabeled.csv').dropna()
df_extra.head()

,id,comment_text
0,0,It's more accurate to call it the M+ (1000) be...
1,1,To be there with a samsung phone\n😂😂😂
2,2,"Thank gosh, a place I can watch it without hav..."
3,3,What happened to the home button on the iPhone...
4,4,Power is the disease. Care is the cure. Keep...


In [26]:
emoji_set = set()

for comment in tqdm(df_train['comment_text']):
    emojies = emoji.emoji_list(comment)
    for e in emojies:
        emoji_set.add(e['emoji'])
        
len(emoji_set)

100%|██████████| 51153/51153 [00:02<00:00, 19019.84it/s]


1343

In [27]:
def text_feature_extraction(df: pd.DataFrame, col: str):
    df['emoji_count'] = df[col].progress_apply(lambda x: len(emoji.emoji_list(x)))
    df['happy'] = df[col].progress_apply(lambda x: x.count(':)') + x.count(';)') + x.count('(:') + x.count('(;'))
    df['sad'] = df[col].progress_apply(lambda x: x.count(':(') + x.count(';(') + x.count('):') + x.count(');'))
    return df

def text_normalize(text: str):
    text = text.lower()
    text = text.replace('\n', ' ')
    text = text.replace('\r', ' ')
    text = text.replace('\t', ' ')
    text = "".join([ch for ch in text if ch not in string.punctuation])
    text = emoji.replace_emoji(text, replace='')

    words = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words]
    
    stemmer = PorterStemmer()
    words = [stemmer.stem(word) for word in words]
    text = ' '.join(words)
    return text
    
text_normalize(df_train['comment_text'][4])

'mustard colour tho ps absolut love video whenev someon mention babi name instantli think sim okayyyyy'

In [ ]:
df_train_clean = df_train.copy()
df_extra_clean = df_extra.copy()

df_train_clean = text_feature_extraction(df_train_clean, 'comment_text')
df_train_clean['comment_text'] = df_train_clean['comment_text'].progress_apply(text_normalize)

# df_extra_clean = text_feature_extraction(df_extra_clean, 'comment_text')
df_extra_clean['comment_text'] = df_extra_clean['comment_text'].progress_apply(text_normalize)

df_train_clean.head()

  0%|          | 0/51153 [00:00<?, ?it/s]

100%|██████████| 652233/652233 [05:33<00:00, 1954.97it/s]


,id,comment_text,target,emoji_count,happy,sad
0,59,eh c kii ce favoritism lui il tou led iphon ap...,1,0,0,0
1,167,myth neanderth stupid actual brainpow human ti...,1,0,0,0
2,169,emma super whitenin news,1,0,0,0
3,372,chelsea aubrey favourit teen mom well especi c...,1,4,0,0
4,373,mustard colour tho ps absolut love video whene...,1,2,1,0


In [30]:
all_text = df_train_clean['comment_text'].tolist() + df_extra_clean['comment_text'].tolist()
all_text[:10]

['eh c kii ce favoritism lui il tou led iphon appl watch ipad macbook en avanc est gratuit wallah je reflechi une ide poir wow ador je sui rich mdrr sa arrivera qau autr',
 'myth neanderth stupid actual brainpow human time got assimil',
 'emma super whitenin news',
 'chelsea aubrey favourit teen mom well especi cole babi',
 'mustard colour tho ps absolut love video whenev someon mention babi name instantli think sim okayyyyy',
 'omg u tag dina never see anyon mention even though she amaz',
 'said your girl twin would call lyla luci sister liter look like omg name lyla',
 'love lousi',
 'im man unit fan anyon cant see hazard best player prem last season mad',
 'christensen speak better english rooney']

In [35]:
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 3), stop_words='english')
tfidf.fit(all_text)

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,analyzer,'word'
,stop_words,'english'
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"


In [42]:
np.array([0, 1, 2]).nonzero()[0].shape

(2,)

In [44]:
X_tfidf = tfidf.transform(df_train_clean['comment_text'])

svd = TruncatedSVD(n_components=10000, random_state=42)
X_tfidf = svd.fit_transform(X_tfidf)

print(f"TF-IDF матрица: {X_tfidf.shape}")
print(f"Ненулевых элементов: {X_tfidf.nonzero()[0].shape[0] / np.prod(X_tfidf.shape) * 100:.2f}%")

KeyboardInterrupt: 